# 01 · Explore the OpenAlex ALAN papers

Matches the title and abstract of each paper against the word lists in `vocab/terms.py`, flags papers that are off-topic for the biodiversity focus (human health), and saves a table of the results.

**Reads:** `data/raw/openalex_alan.csv` (made by `src/fetch_openalex.py`), `kg/seed_statements.csv`, `kg/items.csv`
**Writes:** `data/output/table_abstracts.csv` (git-ignored)

**After editing `vocab/terms.py`:** click Restart, then Run All. Python keeps the old word lists in memory otherwise.

## Progress log

| Date | Change | Papers with no match | In-scope papers with no match |
|---|---|---|---|
| 17 Nov | Round 1 baseline | 949 of 2,227 | not measured yet |
| 17 Nov | + zebrafish, elasmobranch, shark | 938 | not measured yet |
| 17 Nov | + seabird, petrel, Procellariiformes | 937 | 452 of 1,576 (simple flag) |
| 18 Nov | + Drosophila, deer, mouse, mice, rat, lizard, anole, frog, amphipod, chipmunk, epiphyte, wildlife | 804 | 385 of 1,666 (simple flag) |

3-way scope after the 18 Nov terms: in scope 1,666 · off-topic 451 · unsure 110.

Add a row here after each round of word-list changes.

## 1. Setup: load the papers and define the matcher

In [ ]:
import re
import sys
import pandas as pd

sys.path.append("..")
from vocab import terms

df = pd.read_csv("../data/raw/openalex_alan.csv")
df["text"] = df["title"].fillna("") + ". " + df["abstract"].fillna("")

`make_pattern` builds a whole-word regex for each term. Abbreviations (SQM, PAR, lx) must match exactly. Other terms ignore case and also match a simple plural ("moth" matches "moths").

A term has to start at a word boundary, so "bird" does not match inside "seabird" and "fish" does not match inside "zebrafish": list compounds as their own terms. Irregular plurals need their own entry too ("mouse" and "mice").

In [ ]:
def make_pattern(term):
    # Abbreviations like SQM, PAR, W must match exactly (capital letters).
    case_sensitive = term.isupper() or term == "lx"
    if case_sensitive:
        return re.compile(r"(?<!\w)" + re.escape(term) + r"(?!\w)")
    # Other terms ignore case and also match a plural ("moth" -> "moths").
    return re.compile(r"(?<!\w)" + re.escape(term) + r"(?:e?s)?(?!\w)", re.IGNORECASE)


def find_terms(text, term_list):
    found = [term for term in term_list if make_pattern(term).search(text)]
    return "; ".join(found)

## 2. Round 1 matching

One new column per category holds the terms found in each paper's title and abstract. `no_match` is True when a paper matches nothing in any of the five categories.

In [ ]:
categories = {
    "methods":     terms.METHODS_INSTRUMENTS,
    "quantities":  terms.QUANTITIES,
    "units":       terms.UNITS,
    "disciplines": terms.DISCIPLINES,
    "species":     terms.TAXA,
}

# One new column per category: the terms found in each paper
for column, term_list in categories.items():
    df[column] = df["text"].apply(lambda t: find_terms(t, term_list))

# True for papers where nothing matched in any of the five categories
no_match = (df[list(categories)] == "").all(axis=1)
print(no_match.sum(), "of", len(df), "papers have no match")

## 3. Scope: is the paper about biodiversity?

The project follows the Hölker et al. 2021 "11 pressing research questions" paper (biodiversity), so human-health papers are off-topic, unless they are about animals.

**Simple flag (first cell below):** human-health word AND no species match. A flagged paper can still match another category (unit, quantity), so flagged and unmatched overlap.

**3-way `scope` (second cell, refines the simple flag):**
- `off-topic`: flagged, with 2 or more health terms or one strong term (cancer, patient, diabetes, ...)
- `unsure`: flagged on a single weak term (sleep, participant, clinical, ...)
- `in scope`: everything else

In [ ]:
df["human_health"] = df["text"].apply(lambda t: find_terms(t, terms.HUMAN_HEALTH))

off_topic = (df["human_health"] != "") & (df["species"] == "")
in_scope = ~off_topic

print(off_topic.sum(), "papers flagged off-topic (human health, no species)")
print((no_match & in_scope).sum(), "of", in_scope.sum(), "in-scope papers have no match")

In [ ]:
strong = ["cancer", "cardiovascular", "obesity", "diabetes", "patient",
          "COVID", "depression", "mental health", "insomnia", "epidemiology"]

n_terms = df["human_health"].apply(lambda s: len(s.split("; ")) if s else 0)
strong_hits = df["human_health"].apply(
    lambda s: sum(t in strong for t in s.split("; ")))

flagged = (df["human_health"] != "") & (df["species"] == "")

df["scope"] = "in scope"
df.loc[flagged & ((n_terms >= 2) | (strong_hits >= 1)), "scope"] = "off-topic"
df.loc[flagged & (n_terms == 1) & (strong_hits == 0), "scope"] = "unsure"

df["scope"].value_counts()

## 4. Summary numbers

In scope = all papers minus flagged. The flagged group and the no-match group overlap, so the table splits each of them.

In [ ]:
n = len(df)
matched = ~no_match

summary = pd.DataFrame({
    "group": [
        "All papers",
        "Match at least one category",
        "Match nothing",
        "Flagged off-topic (human health, no species)",
        "   flagged AND match nothing",
        "   flagged BUT match something else",
        "In scope (not flagged)",
        "   in scope AND match nothing",
        "   in scope AND match something",
    ],
    "papers": [
        n,
        matched.sum(),
        no_match.sum(),
        off_topic.sum(),
        (off_topic & no_match).sum(),
        (off_topic & matched).sum(),
        in_scope.sum(),
        (in_scope & no_match).sum(),
        (in_scope & matched).sum(),
    ],
})
summary["% of all"] = (summary["papers"] / n * 100).round(1)
summary

## 5. Checking the flag

Reading samples to see whether the flag removes papers it should not. Look for animals, ecology and light-measurement papers.

In [ ]:
df[off_topic][["title", "human_health"]].sample(15, random_state=2)

In [ ]:
n_terms = df["human_health"].apply(lambda s: len(s.split("; ")) if s else 0)
one_term = off_topic & (n_terms == 1)

print(one_term.sum(), "of", off_topic.sum(), "flagged on a single term")
df[one_term]["human_health"].value_counts()

In [ ]:
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 200)

unsure = df[df["scope"] == "unsure"][["title", "human_health"]]
unsure.sort_values("human_health")

**Findings from reading the unsure list (18 Nov, when it had 151 papers):**
- About 23 wildlife and ecology papers were hidden only because their animal was missing from `TAXA` (fixed by adding animal terms).
- About 14 policy, regulation and governance papers (dark-sky governance, lighting standards, public perception): possibly the "units to legal issues" thread.
- A few light-measurement papers (nighttime Earth observation, city light pollution quantification, sensors).
- Duplicate records exist (same title twice): add a de-duplication step.
- Some retrieved papers never mention light (equestrian business, malaria, photography): a "mentions light" flag would catch them.

## 6. Save the table

Run this after the scope step so the saved CSV includes the `human_health` and `scope` columns.

In [ ]:
df.to_csv("../data/output/table_abstracts.csv", index=False)


## Appendix A: seed KG overview (context only)

Not part of the mining flow. Shows how the 40 hand-entered triples in `kg/seed_statements.csv` map onto items, and which items are classified in `kg/items.csv`.

In [ ]:
kg = pd.read_csv("../kg/seed_statements.csv")

items = pd.DataFrame({"item": sorted(set(kg["subject"]) | set(kg["object"]))})

def roles(name):
    as_subject = sorted(set(kg.loc[kg["subject"] == name, "property"]))
    as_object = sorted(set(kg.loc[kg["object"] == name, "property"]))
    return pd.Series({"subject_of": ", ".join(as_subject), "object_of": ", ".join(as_object)})

items = pd.concat([items, items["item"].apply(roles)], axis=1)
print(len(items))
items

In [ ]:
items_cls = pd.read_csv("../kg/items.csv")
print(items_cls.shape)
print(items_cls["class"].value_counts())
known = set(kg["subject"]) | set(kg["object"])
print("not found in seed_statements:", sorted(set(items_cls["item"]) - known))



## Appendix B: scratch snippets (commented out)

Uncomment to use. The first shows a random sample of papers that matched nothing; the second prints full abstracts for chosen row numbers.

In [ ]:
# df[no_match][["title", "abstract"]].sample(10, random_state=1)

In [ ]:
# for i in [144, 1716, 940]:
#     print(i, df.loc[i, "title"])
#     print(df.loc[i, "abstract"])
#     print()